# Movie Intelligence — EDA

Exploratory Data Analysis on enriched TMDB movie dataset (1,986 movies).
Data source: `data/enriched/movies_enriched.parquet`

In [1]:
import sys
print(sys.executable)

c:\Users\ycode\Desktop\2nd_YouCode\briefs\Movie_Intelligence\.venv\Scripts\python.exe


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Style
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 12

FIG_DIR = Path("reports/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Load data
df = pd.read_parquet("../data/enriched/movies_enriched.parquet")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

Shape: (1986, 35)
Columns: ['id', 'imdb_id', 'title', 'original_title', 'original_language', 'overview', 'tagline', 'status', 'release_date', 'runtime', 'budget', 'revenue', 'popularity', 'vote_average', 'vote_count', 'adult', 'video', 'homepage', 'poster_path', 'backdrop_path', 'has_budget', 'has_revenue', 'release_year', 'release_month', 'collection_id', 'collection_name', 'original_language_name', 'profit', 'roi', 'decade', 'runtime_bucket', 'weighted_rating', 'budget_tier', 'revenue_tier', 'search_text']


## 1. Distribution des notes (vote_average) et popularité

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# vote_average
sns.histplot(df["vote_average"].dropna(), bins=30, kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("Distribution des notes moyennes (vote_average)")
axes[0].set_xlabel("Note moyenne (0-10)")
axes[0].axvline(df["vote_average"].mean(), color='red', linestyle='--', label=f'Moyenne: {df["vote_average"].mean():.2f}')
axes[0].legend()

# popularity
sns.histplot(df["popularity"].dropna(), bins=30, kde=True, ax=axes[1], color="coral")
axes[1].set_title("Distribution de la popularité")
axes[1].set_xlabel("Popularité TMDB")
axes[1].set_yscale("log")
axes[1].axvline(df["popularity"].median(), color='red', linestyle='--', label=f'Médiane: {df["popularity"].median():.1f}')
axes[1].legend()

plt.tight_layout()
plt.savefig(FIG_DIR / "1_distribution_notes_popularite.png", dpi=300)
plt.show()

**Interprétation :**
- **vote_average** : Distribution quasi-normale centrée autour de 5.5-6. La plupart des films ont des notes moyennes, peu de très mauvais (<3) ou excellents (>8.5).
- **popularity** : Distribution très asymétrique (log-scale). Quelques blockbusters ont une popularité extrême (>100), la majorité <20. La médiane (~10) est bien plus représentative que la moyenne.

## 2. Films par genre

In [ ]:
# Load genres to count per genre
genres = pd.read_parquet("data/enriched/genres.parquet")
genre_counts = genres["genre_name"].value_counts().sort_values(ascending=True)

plt.figure(figsize=(10, 8))
bars = plt.barh(range(len(genre_counts)), genre_counts.values, color="steelblue")
plt.yticks(range(len(genre_counts)), genre_counts.index)
plt.xlabel("Nombre de films")
plt.title("Nombre de films par genre")

# Add value labels
for i, (idx, val) in enumerate(genre_counts.items()):
    plt.text(val + 5, i, str(val), va='center')

plt.tight_layout()
plt.savefig(FIG_DIR / "2_films_par_genre.png", dpi=300)
plt.show()

**Interprétation :**
- **Drame, Comédie, Action** dominent (600-700 films chacun). Ce sont les genres les plus produits.
- **Documentaire, Téléfilm, Western, Musique** sont rares (<100). Cela reflète la production commerciale dominante sur TMDB.
- Note : un film peut avoir plusieurs genres, donc la somme > 1986.

## 3. Sorties par année (et par décennie)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Par année
year_counts = df["release_year"].dropna().astype(int).value_counts().sort_index()
axes[0].plot(year_counts.index, year_counts.values, color="steelblue", marker='o', markersize=3)
axes[0].set_title("Nombre de films par année")
axes[0].set_xlabel("Année")
axes[0].set_ylabel("Nombre de films")
axes[0].grid(True, alpha=0.3)

# Par décennie
decade_counts = df["decade"].dropna().astype(int).value_counts().sort_index()
axes[1].bar(decade_counts.index.astype(str), decade_counts.values, color="coral")
axes[1].set_title("Nombre de films par décennie")
axes[1].set_xlabel("Décennie")
axes[1].set_ylabel("Nombre de films")
for i, v in enumerate(decade_counts.values):
    axes[1].text(i, v + 5, str(v), ha='center')

plt.tight_layout()
plt.savefig(FIG_DIR / "3_sorties_par_annee_decennie.png", dpi=300)
plt.show()

**Interprétation :**
- **Par année** : Forte croissance à partir des années 1990, pic dans les années 2010-2020. Reflète l'explosion de la production et la couverture TMDB.
- **Par décennie** : 2010 et 2020 concentrent ~60% des films. Les décennies antérieures (1930-1980) sont sous-représentées (films classiques moins bien couverts par TMDB).

## 4. Durée des films (runtime)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogramme
sns.histplot(df["runtime"].dropna(), bins=30, kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("Distribution de la durée des films")
axes[0].set_xlabel("Durée (minutes)")
axes[0].axvline(df["runtime"].median(), color='red', linestyle='--', label=f'Médiane: {df["runtime"].median():.0f} min')
axes[0].legend()

# Boxplot par bucket
order = ["<60", "60-90", "90-120", "120-150", "150-180", ">180"]
sns.boxplot(data=df, x="runtime_bucket", y="vote_average", order=order, ax=axes[1], palette="Set2")
axes[1].set_title("Note moyenne par tranche de durée")
axes[1].set_xlabel("Tranche de durée")
axes[1].set_ylabel("Note moyenne")

plt.tight_layout()
plt.savefig(FIG_DIR / "4_duree_films.png", dpi=300)
plt.show()

**Interprétation :**
- **Distribution** : Pic autour de 90-120 min (format standard). Queue longue vers les longs métrages (>150 min).
- **Note vs durée** : Les films 90-120 min ont les meilleures notes médianes. Les très courts (<60) et très longs (>180) ont plus de variance. Le format standard semble optimal pour la satisfaction spectateur.

## 5. Budget vs Revenus

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter budget vs revenue (log scale)
mask = df["budget"].notna() & df["revenue"].notna() & (df["budget"] > 0) & (df["revenue"] > 0)
scatter = axes[0].scatter(df.loc[mask, "budget"], df.loc[mask, "revenue"], 
                           c=df.loc[mask, "vote_average"], cmap="RdYlGn", alpha=0.6, s=20)
axes[0].set_xscale("log")
axes[0].set_yscale("log")
axes[0].plot([1e4, 3e8], [1e4, 3e8], 'k--', alpha=0.5, label="Break-even")
axes[0].set_title("Budget vs Revenus (échelle log)")
axes[0].set_xlabel("Budget (USD)")
axes[0].set_ylabel("Revenus (USD)")
axes[0].legend()
plt.colorbar(scatter, ax=axes[0], label="Note moyenne")

# Boxplot revenue par budget tier
sns.boxplot(data=df.dropna(subset=["budget_tier", "revenue"]), 
            x="budget_tier", y="revenue", 
            order=["Very Low", "Low", "Medium", "High", "Very High"], 
            ax=axes[1], palette="Blues")
axes[1].set_yscale("log")
axes[1].set_title("Revenus par tranche de budget")
axes[1].set_xlabel("Tranche de budget")
axes[1].set_ylabel("Revenus (USD, log)")
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(FIG_DIR / "5_budget_vs_revenus.png", dpi=300)
plt.show()

**Interprétation :**
- **Scatter** : Forte corrélation (budget élevé → revenus élevés en général). Points au-dessus de la diagonale = rentables. Beaucoup de films à budget modeste font mieux que des blockbusters.
- **Boxplot** : Médiane des revenus augmente avec le budget tier, mais variance énorme dans "Very High" (quelques flops coûteux). Le tier "Medium" a souvent le meilleur ratio risque/récompense.

## 6. Votes vs Popularité

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter vote_count vs popularity
scatter = axes[0].scatter(df["vote_count"], df["popularity"], 
                           c=df["vote_average"], cmap="viridis", alpha=0.6, s=20)
axes[0].set_xscale("log")
axes[0].set_yscale("log")
axes[0].set_title("Nombre de votes vs Popularité")
axes[0].set_xlabel("Nombre de votes (log)")
axes[0].set_ylabel("Popularité (log)")
plt.colorbar(scatter, ax=axes[0], label="Note moyenne")

# Vote average vs vote_count (IMDb style)
axes[1].scatter(df["vote_count"], df["vote_average"], alpha=0.5, s=15, color="steelblue")
axes[1].set_xscale("log")
axes[1].set_title("Note moyenne vs Nombre de votes")
axes[1].set_xlabel("Nombre de votes (log)")
axes[1].set_ylabel("Note moyenne")
# Ligne de référence : moyenne globale
axes[1].axhline(df["vote_average"].mean(), color='red', linestyle='--', 
                label=f'Moyenne globale: {df["vote_average"].mean():.2f}')
axes[1].legend()

plt.tight_layout()
plt.savefig(FIG_DIR / "6_votes_vs_popularite.png", dpi=300)
plt.show()

**Interprétation :**
- **Votes vs Popularité** : Corrélation forte (plus de votes = plus populaire). Les films très notés (vert) ont tendance à être populaires, mais certains films populaires ont des notes moyennes (jaune/orange).
- **Note vs Votes** : Phénomène de "régression vers la moyenne" : films avec peu de votes ont des notes extrêmes (très hautes ou très basses). À partir de ~1000 votes, les notes se stabilisent autour de 5.5-7.5.

## 7. Boxplots et Corrélations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Boxplot: note par genre (top 8)
genres = pd.read_parquet("data/enriched/genres.parquet")
df_genres = df.merge(genres, left_on="id", right_on="movie_id")
top_genres = df_genres["genre_name"].value_counts().head(8).index
sns.boxplot(data=df_genres[df_genres["genre_name"].isin(top_genres)], 
            x="genre_name", y="vote_average", ax=axes[0,0], palette="Set3")
axes[0,0].set_title("Note moyenne par genre (Top 8)")
axes[0,0].tick_params(axis='x', rotation=45)
axes[0,0].set_xlabel("")
axes[0,0].set_ylabel("Note moyenne")

# Boxplot: ROI par budget tier
sns.boxplot(data=df.dropna(subset=["budget_tier", "roi"]), 
            x="budget_tier", y="roi", 
            order=["Very Low", "Low", "Medium", "High", "Very High"], 
            ax=axes[0,1], palette="Reds")
axes[0,1].set_yscale("symlog")
axes[0,1].set_title("ROI par tranche de budget")
axes[0,1].set_xlabel("Tranche de budget")
axes[0,1].set_ylabel("ROI (log symétrique)")
axes[0,1].tick_params(axis='x', rotation=45)

# Heatmap de corrélation
num_cols = ["budget", "revenue", "runtime", "popularity", "vote_average", 
          "vote_count", "profit", "roi", "weighted_rating", "release_year"]
corr = df[num_cols].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, 
            ax=axes[1,0], square=True, cbar_kws={"shrink": 0.8})
axes[1,0].set_title("Matrice de corrélation (variables numériques)")

# Pairplot key variables (sample for speed)
sample = df[num_cols].dropna().sample(min(500, len(df)), random_state=42)
sns.scatterplot(data=sample, x="budget", y="revenue", hue="vote_average", 
                palette="RdYlGn", alpha=0.7, ax=axes[1,1])
axes[1,1].set_xscale("log")
axes[1,1].set_yscale("log")
axes[1,1].set_title("Budget vs Revenue (échantillon, coloré par note)")
axes[1,1].set_xlabel("Budget (log)")
axes[1,1].set_ylabel("Revenus (log)")

plt.tight_layout()
plt.savefig(FIG_DIR / "7_boxplots_correlations.png", dpi=300)
plt.show()

**Interprétation :**
- **Note par genre** : Documentaire, Histoire, Musique ont les médianes les plus hautes. Horreur, Action, Science-Fiction ont plus de variance (beaucoup de blockbusters moyens).
- **ROI par budget** : Films à budget "Very Low" ont le ROI médian le plus élevé (peu de risque, gros retours possibles). Budget "Very High" a ROI médian proche de 1 (break-even) mais queue longue positive.
- **Corrélations** : Budget↔Revenue (0.74), Vote_count↔Popularity (0.89), Vote_average↔Weighted_rating (0.97). ROI faiblement corrélé au budget (-0.12) : gros budget ne garantit pas rentabilité.